## FAISS Vector Store

By: Saksham Bansal

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter,CharacterTextSplitter
from langchain_community.document_loaders import TextLoader,PyPDFLoader
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

#utility
import numpy as np
from typing import List,Dict,Any
import os

C:\Users\kanha\AppData\Local\Temp\ipykernel_24152\1777560102.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader,PyPDFLoader


In [4]:
import warnings
warnings.filterwarnings('ignore')

In [5]:
from langchain_core.prompts import ChatPromptTemplate,PromptTemplate
from langchain_core.runnables import (RunnablePassthrough,)
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage,AIMessage
from langchain_community.vectorstores import FAISS


In [6]:
from langchain_huggingface import HuggingFaceEmbeddings



### Data Ingestion And Parsing

In [7]:
sample_documents= [
    Document(
        page_content="""
        Artificial Intelligence (AI) isthe simulation of human intelligence in machines.
        These systems are designed to think like humans and mimic their actions.
        AI can be categorized into  narrow AI and general AI.
        """,
        metadata={
            "source":"AI Introduction","page":1,"topic":"AI"
        }
    ),
       Document(
        page_content="""
        Machine Learning is a subset of Artificial Intelligence.
        It enables computers to learn patterns from data without being explicitly programmed.
        Common types include supervised, unsupervised, and reinforcement learning.
        """,
        metadata={
            "source": "Machine Learning Basics",
            "page": 2,
            "topic": "Machine Learning"
        }
    ),

    Document(
        page_content="""
        Deep Learning is a branch of machine learning that uses artificial neural networks
        with multiple hidden layers. It is widely used in image recognition,
        speech recognition, and natural language processing.
        """,
        metadata={
            "source": "Deep Learning Guide",
            "page": 3,
            "topic": "Deep Learning"
        }
    ),

    Document(
        page_content="""
        Natural Language Processing (NLP) focuses on enabling computers
        to understand, interpret, and generate human language.
        Applications include translation, chatbots, and sentiment analysis.
        """,
        metadata={
            "source": "NLP Handbook",
            "page": 4,
            "topic": "NLP"
        }
    ),

    Document(
        page_content="""
        Computer Vision enables machines to interpret and understand visual information.
        It is used in facial recognition, autonomous vehicles,
        medical image analysis, and object detection.
        """,
        metadata={
            "source": "Computer Vision Notes",
            "page": 5,
            "topic": "Computer Vision"
        }
    ),

    Document(
        page_content="""
        Reinforcement Learning is a machine learning paradigm where an agent
        learns by interacting with an environment. The agent receives rewards
        or penalties based on its actions and aims to maximize cumulative reward.
        """,
        metadata={
            "source": "Reinforcement Learning",
            "page": 6,
            "topic": "Reinforcement Learning"
        }
    ),

    Document(
        page_content="""
        Generative AI refers to models capable of creating new content such as text,
        images, audio, and code. Large Language Models and diffusion models
        are popular examples of generative AI systems.
        """,
        metadata={
            "source": "Generative AI Overview",
            "page": 7,
            "topic": "Generative AI"
        }
    ),

    Document(
        page_content="""
        Large Language Models (LLMs) are deep learning models trained on massive text datasets.
        They can perform tasks such as question answering, summarization,
        translation, and code generation using natural language prompts.
        """,
        metadata={
            "source": "LLM Fundamentals",
            "page": 8,
            "topic": "LLM"
        }
    ),

    Document(
        page_content="""
        Vector databases store embeddings that represent text, images,
        or other data in high-dimensional space.
        They enable efficient similarity search for retrieval-augmented generation (RAG).
        """,
        metadata={
            "source": "Vector Databases",
            "page": 9,
            "topic": "Vector Database"
        }
    ),

    Document(
        page_content="""
        Retrieval-Augmented Generation (RAG) combines information retrieval
        with large language models. Relevant documents are retrieved from a
        knowledge base and provided as context before generating an answer.
        """,
        metadata={
            "source": "RAG Architecture",
            "page": 10,
            "topic": "RAG"
        }
    )

]

## Text splitting

In [9]:
#lets go
text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=[" "]
)

## split documents into chunks
chunks=text_splitter.split_documents(sample_documents)
print(chunks)

[Document(metadata={'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}, page_content='Artificial Intelligence (AI) isthe simulation of human intelligence in machines.\n        These systems are designed to think like humans and mimic their actions.\n        AI can be categorized into  narrow AI and general AI.'), Document(metadata={'source': 'Machine Learning Basics', 'page': 2, 'topic': 'Machine Learning'}, page_content='Machine Learning is a subset of Artificial Intelligence.\n        It enables computers to learn patterns from data without being explicitly programmed.\n        Common types include supervised, unsupervised, and reinforcement learning.'), Document(metadata={'source': 'Deep Learning Guide', 'page': 3, 'topic': 'Deep Learning'}, page_content='Deep Learning is a branch of machine learning that uses artificial neural networks\n        with multiple hidden layers. It is widely used in image recognition,\n        speech recognition, and natural language processing.'), D

In [10]:
print(chunks[0])

page_content='Artificial Intelligence (AI) isthe simulation of human intelligence in machines.
        These systems are designed to think like humans and mimic their actions.
        AI can be categorized into  narrow AI and general AI.' metadata={'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}


In [11]:
print(len(chunks))

10


## Load Embedding models


In [12]:
##embeddings
embeddings=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
embeddings

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [13]:
texts=["AI","Machine Learning","deep Learning","NLP"]
batch_embeddings=embeddings.embed_documents(texts)
print(batch_embeddings[0])

[-0.03653926029801369, -0.015164357610046864, 0.016432464122772217, 0.010568826459348202, 0.006010559853166342, -0.018473288044333458, 0.08546527475118637, 0.02096845768392086, 0.027815353125333786, 0.012431694194674492, -0.02937648445367813, -0.031135285273194313, 0.03491251543164253, -0.018150851130485535, -0.06498481333255768, 0.0516824871301651, -0.019606219604611397, -0.015734203159809113, -0.13371676206588745, -0.09645991772413254, -0.02547178603708744, -0.0014895843341946602, -0.006349374074488878, -0.02582065388560295, -0.02737179957330227, 0.12268992513418198, -0.007792469579726458, -0.03852269425988197, 0.014383504167199135, -0.09218426793813705, 0.008695731870830059, 0.00261337636038661, 0.09103471785783768, -0.030313612893223763, -0.09604638814926147, 0.022289087995886803, -0.09024307876825333, -0.032947368919849396, 0.0715833380818367, -0.008893106132745743, -0.025708934292197227, -0.0791396051645279, 0.014530384913086891, -0.07420430332422256, 0.08045009523630142, 0.07804

In [14]:
## Compare Embedding using cosine
def compare_embeddings(text1:str,text2:str):
    "Compare"
    emb1=np.array(embeddings.embed_query(text1))
    emb2=np.array(embeddings.embed_query(text2))

    similiarity=np.dot(emb1,emb2)
    return similiarity

In [16]:
print(f"AI vs Artificial Intelligence:{compare_embeddings('AI','Artificial Intelligence')}")

AI vs Artificial Intelligence:0.7912462012966478


## Create FAISS

In [17]:
vectorstore=FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)
print(f"Vector Store created with {vectorstore.index.ntotal} vectors")

Vector Store created with 10 vectors


In [18]:
vectorstore

In [19]:
## Save vector store for future
vectorstore.save_local("faiss_index")
print("vectorstore saved to'faiss_index directory")


vectorstore saved to'faiss_index directory


In [20]:
## Load vector store
loaded_vectorstore=FAISS.load_local(
    "faiss_index",
    embeddings=embeddings,
    allow_dangerous_deserialization=True
)
print(f"Loaded vector store contains {loaded_vectorstore.index.ntotal} vectors ")

Loaded vector store contains 10 vectors 


In [21]:
## Similiarity search
query="What is deep learning"
results=vectorstore.similarity_search(query,k=3)
print(results)

[Document(id='7e8a4035-7062-4b6a-bc17-54d905f82ee4', metadata={'source': 'Deep Learning Guide', 'page': 3, 'topic': 'Deep Learning'}, page_content='Deep Learning is a branch of machine learning that uses artificial neural networks\n        with multiple hidden layers. It is widely used in image recognition,\n        speech recognition, and natural language processing.'), Document(id='75b7f9d7-2d00-4aed-8406-80d961165f2a', metadata={'source': 'Machine Learning Basics', 'page': 2, 'topic': 'Machine Learning'}, page_content='Machine Learning is a subset of Artificial Intelligence.\n        It enables computers to learn patterns from data without being explicitly programmed.\n        Common types include supervised, unsupervised, and reinforcement learning.'), Document(id='9bd718e3-d46c-4365-a77e-8afea7443d9c', metadata={'source': 'LLM Fundamentals', 'page': 8, 'topic': 'LLM'}, page_content='Large Language Models (LLMs) are deep learning models trained on massive text datasets.\n        Th

In [24]:
print(f"query:{query}")
print("Top 3 Similiar search:")
for i,doc in enumerate(results):
    print(f"\n Q{i+1}) Source:{doc.metadata['source']}")
    print(f"   Content:{doc.page_content[:200]}....")

query:What is deep learning
Top 3 Similiar search:

 Q1) Source:Deep Learning Guide
   Content:Deep Learning is a branch of machine learning that uses artificial neural networks
        with multiple hidden layers. It is widely used in image recognition,
        speech recognition, and natural ....

 Q2) Source:Machine Learning Basics
   Content:Machine Learning is a subset of Artificial Intelligence.
        It enables computers to learn patterns from data without being explicitly programmed.
        Common types include supervised, unsuperv....

 Q3) Source:LLM Fundamentals
   Content:Large Language Models (LLMs) are deep learning models trained on massive text datasets.
        They can perform tasks such as question answering, summarization,
        translation, and code generati....


In [29]:
query="What is deep learning"
results_score=vectorstore.similarity_search_with_score(query,k=3)
print(results_score)

[(Document(id='7e8a4035-7062-4b6a-bc17-54d905f82ee4', metadata={'source': 'Deep Learning Guide', 'page': 3, 'topic': 'Deep Learning'}, page_content='Deep Learning is a branch of machine learning that uses artificial neural networks\n        with multiple hidden layers. It is widely used in image recognition,\n        speech recognition, and natural language processing.'), np.float32(0.27004898)), (Document(id='75b7f9d7-2d00-4aed-8406-80d961165f2a', metadata={'source': 'Machine Learning Basics', 'page': 2, 'topic': 'Machine Learning'}, page_content='Machine Learning is a subset of Artificial Intelligence.\n        It enables computers to learn patterns from data without being explicitly programmed.\n        Common types include supervised, unsupervised, and reinforcement learning.'), np.float32(1.0255768)), (Document(id='9bd718e3-d46c-4365-a77e-8afea7443d9c', metadata={'source': 'LLM Fundamentals', 'page': 8, 'topic': 'LLM'}, page_content='Large Language Models (LLMs) are deep learning 

In [30]:
print(f"query:{query}")
print("Top 3 Similiar search:")
for i,doc in enumerate(results_score):
    print(f"\n Q{i+1}) Source:{doc.metadata['source']}")
    print(f"   Content:{doc.page_content[:200]}....")

query:What is deep learning
Top 3 Similiar search:


AttributeError: 'tuple' object has no attribute 'metadata'

## Build RAG Chain with LCEL

In [ ]:
from langchain.chat_models import init_chat_model
API_KEY=""
llm=init_chat_model(model="groq:llama-3.3-70b-versatile",api_key=API_KEY )

In [60]:
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001D8FB154D50>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001D8FB154B50>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [61]:
simple_prompt=ChatPromptTemplate.from_template("""Answer the question based only on following context:
Context:{context}
Question:{question}
Answer:""")

In [62]:
retriever=vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={'k':3}
)
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001D8C8F3DF90>, search_kwargs={'k': 3})

In [63]:
def format_doc(docs:List[Document])->str:
    formatted=[]
    for i,doc in enumerate(docs):
        source=doc.metadata.get('source','Unknown')
        formatted.append(f"Document {i+1} (Source:{source}):\n{doc.page_content}")
    return "\n\n".join(formatted)

In [64]:
simple_rag_chain=(
    {"context":retriever|format_doc,"question":RunnablePassthrough()}|simple_prompt|llm|StrOutputParser()
)

In [65]:
simple_rag_chain

{
  context: VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001D8C8F3DF90>, search_kwargs={'k': 3})
           | RunnableLambda(format_doc),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based only on following context:\nContext:{context}\nQuestion:{question}\nAnswer:'), additional_kwargs={})])
| ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False,

In [66]:
## conversational rag chain
conversational_prompt=ChatPromptTemplate.from_messages([
    ("system","You are a helpful AI asisstant. Use the providedd context to aanswer the qustions."),
    ('placeholder',"{chat_history}"),
    ("human","Context:{context}\n\nQuestion:{Input}")
])

In [ ]:
conversational_rag=RunnablePassthrough.assign({(
    context=lambda x:format_doc(retriever.invoke(x["input"]))
)|conversational_prompt|llm|StrOutputParser
}
)

In [68]:
conversational_rag

RunnableAssign(mapper={
  context: RunnableLambda(lambda x: format_doc(retriever.invoke(x['input'])))
})
| ChatPromptTemplate(input_variables=['Input', 'context'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='

In [73]:
def test_rag_chain(question:str):
    print(f"Question {question}")
    print("."*80)

    print("\n simple rag chain")
    answer=simple_rag_chain.invoke(question)
    print(f"Answer:{answer}")
    print(f"Question {question}")
    print("."*80)

    print("\n conversational rag chain")
    answer=conversational_rag.invoke([question])
    print(f"Answer:{answer}")



In [74]:
test_rag_chain(query)

Question What is deep learning
................................................................................

 simple rag chain
Answer:Deep Learning is a branch of machine learning that uses artificial neural networks with multiple hidden layers. It is widely used in image recognition, speech recognition, and natural language processing.
Question What is deep learning
................................................................................

 conversational rag chain


ValueError: The input to RunnablePassthrough.assign() must be a dict.